In [10]:
# Load the raw PM2.5 dataset and required libraries
from pathlib import Path

import pandas as pd
import requests

project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "air_quality.csv"

df = pd.read_csv(data_path, parse_dates=["date"])

print("Shape:", df.shape)
df.head()

Shape: (3495, 2)


,date,pm25
0,2016-11-09 00:00:00+05:30,270.0
1,2016-11-10 00:00:00+05:30,259.0
2,2016-11-11 00:00:00+05:30,274.0
3,2016-11-12 00:00:00+05:30,238.0
4,2016-11-13 00:00:00+05:30,175.0


In [2]:
# Create a continuous daily time series and expose missing observations
df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()

full_dates = pd.date_range(
    start=df["date"].min(),
    end=df["date"].max(),
    freq="D"
)

df = (
    df.set_index("date")
      .reindex(full_dates)
      .rename_axis("date")
      .reset_index()
)

print("Rows after creating full timeline:", len(df))
print("Missing PM2.5 values:", df["pm25"].isna().sum())

Rows after creating full timeline: 3607
Missing PM2.5 values: 204


In [3]:
# Find consecutive gaps in the PM2.5 time series
missing = df["pm25"].isna()

gap_id = missing.ne(missing.shift()).cumsum()

missing_gaps = (
    df[missing]
    .groupby(gap_id[missing])
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        gap_days=("date", "size")
    )
    .sort_values("gap_days", ascending=False)
    .reset_index(drop=True)
)

print("Number of missing gaps:", len(missing_gaps))
print("Largest gap:", missing_gaps["gap_days"].max(), "days")

missing_gaps.head(10)

Number of missing gaps: 35
Largest gap: 31 days


,start_date,end_date,gap_days
0,2019-02-12,2019-03-14,31
1,2023-02-17,2023-03-18,30
2,2022-05-11,2022-06-03,24
3,2017-07-22,2017-08-10,20
4,2017-12-07,2017-12-26,20
5,2017-08-26,2017-09-06,12
6,2023-02-03,2023-02-13,11
7,2021-07-26,2021-08-02,8
8,2026-04-29,2026-05-03,5
9,2024-04-05,2024-04-08,4


In [4]:
# Fill only short PM2.5 gaps of up to 2 consecutive days
df["pm25"] = df["pm25"].interpolate(
    method="linear",
    limit=2,
    limit_area="inside"
)

print("Missing PM2.5 after short-gap interpolation:", df["pm25"].isna().sum())

Missing PM2.5 after short-gap interpolation: 146


In [5]:
# Check PM2.5 values for invalid readings
print("Minimum PM2.5:", df["pm25"].min())
print("Maximum PM2.5:", df["pm25"].max())
print("Negative values:", (df["pm25"] < 0).sum())
print("Remaining missing values:", df["pm25"].isna().sum())

Minimum PM2.5: -7.26
Maximum PM2.5: 1990.0
Negative values: 5
Remaining missing values: 146


In [6]:
# Replace physically impossible negative PM2.5 readings with missing values
df.loc[df["pm25"] < 0, "pm25"] = pd.NA

print("Negative values:", (df["pm25"] < 0).sum())
print("Missing PM2.5 values:", df["pm25"].isna().sum())

Negative values: 0
Missing PM2.5 values: 151


In [7]:
# Check the remaining missing PM2.5 gap sizes
missing = df["pm25"].isna()
gap_id = missing.ne(missing.shift()).cumsum()

remaining_gaps = (
    df[missing]
    .groupby(gap_id[missing])
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        gap_days=("date", "size")
    )
    .sort_values("gap_days", ascending=False)
    .reset_index(drop=True)
)

print("Remaining missing gaps:", len(remaining_gaps))
print("Largest remaining gap:", remaining_gaps["gap_days"].max(), "days")

remaining_gaps.head(10)

Remaining missing gaps: 13
Largest remaining gap: 30 days


,start_date,end_date,gap_days
0,2019-02-14,2019-03-15,30
1,2023-02-19,2023-03-18,28
2,2022-05-13,2022-06-03,22
3,2017-12-09,2017-12-26,18
4,2017-07-24,2017-08-10,18
5,2017-08-28,2017-09-06,10
6,2023-02-05,2023-02-13,9
7,2021-07-28,2021-08-02,6
8,2026-05-01,2026-05-03,3
9,2018-09-12,2018-09-13,2


In [8]:
# Save the cleaned PM2.5 time series for later processing
output_path = project_root / "data" / "processed" / "air_quality_clean.csv"

df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df))
print("Missing PM2.5:", df["pm25"].isna().sum())

Saved: c:\Users\aayus\OneDrive\Desktop\Data Science Project\Air Quality Trend & Forecast\data\processed\air_quality_clean.csv
Rows: 3607
Missing PM2.5: 151


In [11]:
# Download daily historical weather data for Delhi
weather_url = "https://archive-api.open-meteo.com/v1/archive"

weather_params = {
    "latitude": 28.6139,
    "longitude": 77.2090,
    "start_date": "2016-11-09",
    "end_date": "2026-09-24",
    "daily": [
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_mean"
    ],
    "timezone": "Asia/Kolkata"
}

response = requests.get(weather_url, params=weather_params, timeout=60)
response.raise_for_status()

weather_data = response.json()["daily"]

weather_df = pd.DataFrame({
    "date": pd.to_datetime(weather_data["time"]),
    "temperature": weather_data["temperature_2m_mean"],
    "humidity": weather_data["relative_humidity_2m_mean"],
    "rainfall": weather_data["precipitation_sum"],
    "wind_speed": weather_data["wind_speed_10m_mean"]
})

print("Shape:", weather_df.shape)
weather_df.head()

Shape: (3607, 5)


,date,temperature,humidity,rainfall,wind_speed
0,2016-11-09,21.7,51,0.0,9.5
1,2016-11-10,21.0,59,0.0,7.1
2,2016-11-11,21.0,64,0.0,7.4
3,2016-11-12,21.7,60,0.0,7.6
4,2016-11-13,21.9,58,0.0,4.9


In [12]:
# Check weather data quality and save the raw dataset
print("Date range:", weather_df["date"].min(), "to", weather_df["date"].max())
print("\nMissing values:")
print(weather_df.isna().sum())

weather_path = project_root / "data" / "raw" / "weather.csv"
weather_df.to_csv(weather_path, index=False)

print("\nSaved:", weather_path)

Date range: 2016-11-09 00:00:00 to 2026-09-24 00:00:00

Missing values:
date           0
temperature    0
humidity       0
rainfall       0
wind_speed     0
dtype: int64

Saved: c:\Users\aayus\OneDrive\Desktop\Data Science Project\Air Quality Trend & Forecast\data\raw\weather.csv


In [13]:
# Load the NASA FIRMS key securely from the .env file
import os
from dotenv import load_dotenv

load_dotenv(project_root / ".env")

NASA_KEY = os.getenv("NASA_FIRMS_MAP_KEY")

print("NASA FIRMS key loaded:", NASA_KEY is not None)

NASA FIRMS key loaded: True


In [14]:
# Test the FIRMS connection and check available satellite datasets
availability_url = (
    f"https://firms.modaps.eosdis.nasa.gov/api/"
    f"data_availability/csv/{NASA_KEY}/ALL"
)

availability_df = pd.read_csv(availability_url)

print("NASA FIRMS connection successful.")
availability_df

NASA FIRMS connection successful.


,data_id,min_date,max_date
0,MODIS_NRT,2026-07-01,2026-09-26
1,MODIS_SP,2000-11-01,2026-06-30
2,VIIRS_NOAA20_NRT,2026-07-01,2026-09-26
3,VIIRS_NOAA20_SP,2018-04-01,2026-06-30
4,VIIRS_NOAA21_NRT,2024-01-17,2026-09-26
5,VIIRS_SNPP_NRT,2026-07-01,2026-09-25
6,VIIRS_SNPP_SP,2012-01-20,2026-06-30
7,LANDSAT_NRT,2022-06-20,2026-09-26
8,GOES_NRT,2022-08-09,2026-09-26
9,BA_MODIS,2000-11-01,2026-07-01


In [15]:
# Test NASA FIRMS historical fire data for the region around Delhi
fire_source = "VIIRS_SNPP_SP"

# west, south, east, north
fire_area = "73,27,79,32"

test_url = (
    f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{NASA_KEY}/{fire_source}/{fire_area}/5/2024-11-01"
)

fire_test_df = pd.read_csv(test_url)

print("Fire detections:", len(fire_test_df))
fire_test_df.head()

Fire detections: 1918


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,27.02002,78.66762,336.75,0.58,0.52,2024-11-01,825,N,VIIRS,n,2,307.43,1.75,D,0
1,27.14807,78.90797,345.98,0.60,0.53,2024-11-01,825,N,VIIRS,n,2,306.84,7.47,D,0
2,27.47755,77.46600,330.14,0.50,0.49,2024-11-01,826,N,VIIRS,n,2,303.45,0.99,D,0
3,27.60803,77.52423,341.38,0.50,0.49,2024-11-01,826,N,VIIRS,n,2,305.36,3.27,D,0
4,27.64535,77.42896,334.39,0.50,0.49,2024-11-01,826,N,VIIRS,l,2,305.25,3.12,D,0


In [16]:
# Inspect the columns needed for fire activity analysis
print("Columns:")
print(fire_test_df.columns.tolist())

print("\nDate range:")
print(fire_test_df["acq_date"].min(), "to", fire_test_df["acq_date"].max())

print("\nConfidence values:")
print(fire_test_df["confidence"].value_counts())

Columns:
['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type']

Date range:
2024-11-01 to 2024-11-02

Confidence values:
confidence
n    1659
l     253
h       6
Name: count, dtype: int64


In [22]:
# Keep only the successfully retrieved historical fire observations
fire_df["date"] = pd.to_datetime(fire_df["acq_date"])

fire_df = (
    fire_df[
        (fire_df["date"] >= "2016-11-09") &
        (fire_df["date"] <= "2023-12-17")
    ]
    .copy()
)

print("Fire detections:", len(fire_df))
print("Date range:", fire_df["date"].min(), "to", fire_df["date"].max())

Fire detections: 843897
Date range: 2016-11-09 00:00:00 to 2023-12-17 00:00:00


In [23]:
# Aggregate satellite detections into daily fire activity
daily_fire = (
    fire_df.groupby("date")
    .agg(
        fire_count=("latitude", "size"),
        total_frp=("frp", "sum"),
        mean_frp=("frp", "mean")
    )
    .reset_index()
    .sort_values("date")
)

fire_path = project_root / "data" / "raw" / "fire_data.csv"
daily_fire.to_csv(fire_path, index=False)

print("Daily fire records:", len(daily_fire))
print("Date range:", daily_fire["date"].min(), "to", daily_fire["date"].max())

Daily fire records: 2519
Date range: 2016-11-09 00:00:00 to 2023-12-17 00:00:00


In [26]:
# Load the saved cleaned PM2.5 data and merge all three data sources
clean_path = project_root / "data" / "processed" / "air_quality_clean.csv"

air_df = pd.read_csv(clean_path, parse_dates=["date"])

air_df["date"] = pd.to_datetime(air_df["date"]).dt.normalize()
weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.normalize()
daily_fire["date"] = pd.to_datetime(daily_fire["date"]).dt.normalize()

model_df = (
    air_df.merge(weather_df, on="date", how="left")
          .merge(daily_fire, on="date", how="left")
)

fire_columns = ["fire_count", "total_frp", "mean_frp"]
fire_period = model_df["date"] <= daily_fire["date"].max()

model_df.loc[fire_period, fire_columns] = (
    model_df.loc[fire_period, fire_columns].fillna(0)
)

print("Shape:", model_df.shape)
print("\nMissing values:")
print(model_df.isna().sum())

Shape: (3607, 9)

Missing values:
date              0
pm25            151
temperature       0
humidity          0
rainfall          0
wind_speed        0
fire_count     1012
total_frp      1012
mean_frp       1012
dtype: int64


In [27]:
# Save the combined dataset for EDA and feature engineering
output_path = project_root / "data" / "processed" / "model_data.csv"

model_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", model_df.shape)
print("Date range:", model_df["date"].min(), "to", model_df["date"].max())

Saved: c:\Users\aayus\OneDrive\Desktop\Data Science Project\Air Quality Trend & Forecast\data\processed\model_data.csv
Shape: (3607, 9)
Date range: 2016-11-09 00:00:00 to 2026-09-24 00:00:00
